# Research Ergonomics Quickstart

This notebook demonstrates the canonical quickstart path for the approved `Research Ergonomics` slice.


In [ ]:
from __future__ import annotations

from quantleet.backtest import BacktestEngine
from quantleet.data import BarSeries, DataFrameDataSource, TimeBar
from quantleet.research import qc, ta
from quantleet.strategy import Strategy
from quantleet.trading.domain.costs import CostConfig

source = DataFrameDataSource(
    frame=[
        {
            "timestamp": "1970-01-01T00:01:00+00:00",
            "open": 5.0,
            "high": 5.0,
            "low": 5.0,
            "close": 5.0,
            "volume": 1.0,
        },
        {
            "timestamp": "1970-01-01T00:02:00+00:00",
            "open": 4.0,
            "high": 4.0,
            "low": 4.0,
            "close": 4.0,
            "volume": 1.0,
        },
        {
            "timestamp": "1970-01-01T00:03:00+00:00",
            "open": 1.0,
            "high": 1.0,
            "low": 1.0,
            "close": 1.0,
            "volume": 1.0,
        },
        {
            "timestamp": "1970-01-01T00:04:00+00:00",
            "open": 10.0,
            "high": 10.0,
            "low": 10.0,
            "close": 10.0,
            "volume": 1.0,
        },
        {
            "timestamp": "1970-01-01T00:05:00+00:00",
            "open": 11.0,
            "high": 11.0,
            "low": 11.0,
            "close": 11.0,
            "volume": 1.0,
        },
    ],
    symbol="BTC/USDT",
    timeframe="1m",
)

bars = source.load()
manual_bars = BarSeries(
    symbol="BTC/USDT",
    timeframe="1m",
    bar_type="time",
    rows=(
        TimeBar(timestamp=60_000, open=5.0, high=5.0, low=5.0, close=5.0, volume=1.0),
        TimeBar(timestamp=120_000, open=4.0, high=4.0, low=4.0, close=4.0, volume=1.0),
    ),
)

costs = CostConfig(tick_size=1.0, slippage_ticks=1.0, fee_rate=0.001)
engine = BacktestEngine(initial_cash=1_000.0, costs=costs)

In [ ]:
class SmaCrossStrategy(Strategy):
    def init(self) -> None:
        self.fast = ta.sma(self.data.close, length=2)
        self.slow = ta.sma(self.data.close, length=3)

    def on_bar(self, bar) -> None:
        if qc.crossover(self.fast, self.slow):
            self.buy(quantity=1)
        elif qc.crossunder(self.fast, self.slow):
            self.sell(quantity=1)


sma_result = engine.run(
    source=source,
    strategy=SmaCrossStrategy,
    label="sma-cross",
)

materialized_result = engine.run(
    bars=bars,
    strategy=SmaCrossStrategy,
)

manual_result = engine.run(
    bars=manual_bars,
    strategy=SmaCrossStrategy,
)

sma_result.report
sma_result.summary
sma_result.trade_log
sma_fig = sma_result.plot()
sma_fig

In [ ]:
class Rsi3070Strategy(Strategy):
    def init(self) -> None:
        self.rsi = ta.rsi(self.data.close, length=2)

    def on_bar(self, bar) -> None:
        if (not self.position.is_open) and not qc.is_na(self.rsi[0]) and self.rsi[0] < 30:
            self.buy(quantity=1)
        elif self.position.is_open and not qc.is_na(self.rsi[0]) and self.rsi[0] > 70:
            self.sell(quantity=1)


rsi_result = engine.run(
    source=source,
    strategy=Rsi3070Strategy,
    label="rsi-30-70",
)

rsi_result.report
rsi_result.equity_curve
rsi_result.drawdown_curve
rsi_result.trade_log